### **Installs and Imports**

In [1]:
!pip install -q --upgrade transformers datasets peft trl

# Upgrade torchao to a compatible version
!pip install --upgrade torchao

import torch, random, re
import requests
import pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model, PeftModel
from datasets import load_dataset
from trl import DPOTrainer, DPOConfig, GRPOTrainer, GRPOConfig

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 121.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 42.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.8/775.8 kB 55.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 889.0/889.0 kB 63.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 69.6 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


### **Load the base model** and tokenizer, put the model on the GPU.

In [2]:
model_id = "HuggingFaceTB/SmolLM-135M"     # BASE, not -Instruct

device    = 'cuda' if torch.cuda.is_available() else 'cpu'
tokenizer = AutoTokenizer.from_pretrained(model_id)
model     = AutoModelForCausalLM.from_pretrained(model_id).to(device)

print(f"Loaded {model_id} on {device} — {sum(p.numel() for p in model.parameters()):,} params")

config.json:   0%|          | 0.00/724 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.69k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/801k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/831 [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  538MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

Loaded HuggingFaceTB/SmolLM-135M on cuda — 134,515,008 params


### **Wrap the Model with LoRA Adapters:**

In [3]:
lora_config = LoraConfig(
    task_type      = "CAUSAL_LM",                # tells peft this is a next-token LM
    r              = 8,                          # the rank — size of A and B
    lora_alpha     = 16,                         # scaling: effective ΔW = (alpha/r)·B·A
    target_modules = ["q_proj", "v_proj"],       # WHICH layers get adapters
    lora_dropout   = 0.05,                       # dropout on the adapter path
    bias           = "none",                     # don't train bias terms
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 460,800 || all params: 134,975,808 || trainable%: 0.3414


### **CPT-LoRA** using HuggingFace wikimedia-dataset:

In [4]:
# CPT source: CLEAN Wikipedia prose (markdown stripped, no Moses artifacts).
ds = load_dataset("wikimedia/wikipedia", "20231101.en",
                  split="train", streaming=True)

num_articles = 4000
articles = []
for i, ex in enumerate(ds):
    if i >= num_articles:
        break
    if ex["text"].strip():
        articles.append(ex["text"].strip())

split      = int(0.9 * len(articles))
train_text = "\n\n".join(articles[:split])
val_text   = "\n\n".join(articles[split:])

# Pack into one flat token stream — identical to before, cleaner source
train_ids = torch.tensor(tokenizer(train_text, add_special_tokens=False)["input_ids"])
val_ids   = torch.tensor(tokenizer(val_text,   add_special_tokens=False)["input_ids"])
print(f"articles: {len(articles)} | train tokens: {len(train_ids):,} | val tokens: {len(val_ids):,}")

README.md:   0%|          | 0.00/131k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/41 [00:00<?, ?it/s]

articles: 4000 | train tokens: 14,948,819 | val tokens: 486,094


### **Scan & Sanity-Check:**

In [5]:
# ─── Sanity-check the CPT corpus for WikiText-style artifacts ───────────
import re, random

# 1. The specific patterns WikiText injected — we want ZERO hits on these.
patterns = {
    "@-@ / @.@ / @,@ markers":  r"@[-.,]@",
    "= = header markers":        r"(?m)^\s*=+.*=+\s*$",
    "space-before-punctuation":  r"\s[.,;:!?](?:\s|$)",   # ' .' ' ,' etc.
    "<unk> tokens":              r"<unk>",
}

full = "\n\n".join(articles)
print(f"Scanning {len(articles)} articles ({len(full):,} chars)\n")

for label, pat in patterns.items():
    hits = re.findall(pat, full)
    flag = "✅ clean" if len(hits) == 0 else f"⚠️ {len(hits):,} hits"
    print(f"{flag:<16} {label}")
    if hits:                                  # show a few examples if found
        for ex in hits[:5]:
            print(f"                 e.g. {ex!r}")

# 2. Character sanity — what's actually in the corpus?
import collections
non_ascii = collections.Counter(c for c in full if ord(c) > 127)
print(f"\nMost common non-ASCII chars (accents/quotes are fine, gibberish is not):")
for ch, n in non_ascii.most_common(15):
    print(f"   {ch!r} (U+{ord(ch):04X}): {n:,}")

# 3. Eyeball three random samples — the ultimate check
print("\n" + "="*70 + "\n  THREE RANDOM SAMPLES — read these\n" + "="*70)
for art in random.sample(articles, 3):
    print("\n" + art[:400].strip() + " …\n" + "-"*70)

Scanning 4000 articles (65,092,842 chars)

✅ clean          @-@ / @.@ / @,@ markers
✅ clean          = = header markers
⚠️ 12,250 hits   space-before-punctuation
                 e.g. ' . '
                 e.g. ' . '
                 e.g. ' . '
                 e.g. ' .\n'
                 e.g. ' , '
✅ clean          <unk> tokens

Most common non-ASCII chars (accents/quotes are fine, gibberish is not):
   '–' (U+2013): 53,469
   '\xa0' (U+00A0): 13,182
   'é' (U+00E9): 7,047
   '—' (U+2014): 5,474
   'á' (U+00E1): 3,505
   'ü' (U+00FC): 2,751
   'í' (U+00ED): 2,456
   'ö' (U+00F6): 2,224
   'ā' (U+0101): 2,184
   'ó' (U+00F3): 1,832
   'ı' (U+0131): 1,332
   '°' (U+00B0): 1,189
   '−' (U+2212): 1,092
   'ç' (U+00E7): 1,062
   'è' (U+00E8): 956

  THREE RANDOM SAMPLES — read these

WADL (channel 38) is a television station licensed to Mount Clemens, Michigan, United States, serving the Detroit area as an affiliate of MyNetworkTV. Locally owned by the Adell Broadcasting Corporation, the

### **Hyperparameters + The Batch Loader:**

In [6]:
block_size, batch_size = 256, 8
lr, weight_decay, grad_clip = 2e-4, 0.01, 1.0
max_steps, eval_every = 500, 50

def get_batch(split):
    d  = train_ids if split == "train" else val_ids
    ix = torch.randint(len(d) - block_size, (batch_size,))
    xb = torch.stack([d[i:i+block_size] for i in ix])
    return xb.to(device)

### **Optimizer** + fixed held-out eval:

In [7]:
optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

@torch.no_grad()
def estimate_loss(batches=20):
    model.eval()
    total = 0.0
    for _ in range(batches):
        xb = get_batch("val")
        total += model(input_ids=xb, labels=xb).loss.item()
    model.train()
    return total / batches

### **Training loop:**

In [8]:
model.train()
for step in range(max_steps):
    xb   = get_batch("train")
    loss = model(input_ids=xb, labels=xb).loss

    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
    optimizer.step()

    if step % eval_every == 0 or step == max_steps - 1:
        print(f"step {step:4d} | train {loss.item():.4f} | val {estimate_loss():.4f}")

step    0 | train 2.6966 | val 3.0219
step   50 | train 3.0646 | val 3.0206
step  100 | train 2.8470 | val 2.9867
step  150 | train 2.7270 | val 3.0395
step  200 | train 2.9275 | val 3.0365
step  250 | train 2.9424 | val 3.0430
step  300 | train 2.9314 | val 3.0160
step  350 | train 2.6087 | val 2.9810
step  400 | train 3.0164 | val 2.9923
step  450 | train 2.7062 | val 2.9815
step  499 | train 2.5699 | val 2.9155


### **Generate:**

In [9]:
def sample(prompt, max_new_tokens=120):
    model.eval()
    enc = tokenizer(prompt, return_tensors="pt").to(device)
    out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=True,
                         temperature=0.7, top_p=0.9, repetition_penalty=1.3,
                         no_repeat_ngram_size=3, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0], skip_special_tokens=True)

print(sample("The history of the Roman Empire"))

The history of the Roman Empire began in 285 BC with the death of Julius Caesar. It was followed by another two hundred years when Augustus, who had been emperor for just three months and then died shortly afterwards after a series of military defeats against native Germanic tribes that included a civil war between the Senate and the people as well as other minor revolts (such as the one which led to the death penalty being used). The first emperors were all descended from the Roman patricians.
During this period, Rome's most famous historian, Livy, wrote about how "the whole empire has become united under the head


### Before **LoRA-CPT**:

The history of the Roman Empire began in 286 BC, when a large group of people from what is now Turkey were expelled by the Romans. They settled around the city of Rome and became known as the "Romans". The Romans had many different cultures that influenced their way of life - Greek culture was very important to them because it helped shape how they saw themselves today!
One interesting thing about the Romans' relationship with other ancient civilizations like Greece comes up again: there are some similarities between these two groups but also lots more differences too (like language). For example; while Greeks spoke Latin instead of Ancient Greek due its

### After **LoRA-CPT**:

The history of the Roman Empire is a fascinating one. It was not only an empire , it also had its own distinct culture . The Romans were very successful in their conquest and rule over much of Europe until they fell under the power of Germanic tribes at the end of the 5th century AD ; however , as well as some of the smaller empires which arose around this time such as the Byzantine Empire ( which included parts of what are now Turkey, Greece and Bulgaria ) , there has been little historical research on the Romans themselves during this period - despite being thought to have originated from the Roman city of Aquileia in Italy where


### **Save The Model Adapters:**

In [10]:
model.save_pretrained("smollm-lora-cpt-wikitext")   # saves ONLY the adapters — a few MB, not 500MB



---

## **LoRA-SFT** using Alpaca Datset from HuggingFace:

---



### **Reload and add the adapters:**

In [11]:
base  = AutoModelForCausalLM.from_pretrained("HuggingFaceTB/SmolLM-135M").to(device)
model = PeftModel.from_pretrained(base, "smollm-lora-cpt-wikitext", is_trainable=True).to(device)
model.print_trainable_parameters()   # should say ~460,800 trainable — NOT 0

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

trainable params: 460,800 || all params: 134,975,808 || trainable%: 0.3414


### **Load Dataset** (load + format the SFT data (instruction/response pairs)):

In [12]:
ds = load_dataset("tatsu-lab/alpaca")          # only a 'train' split exists

def format_pair(row):
    instr, inp, out = row["instruction"], row["input"], row["output"]
    if inp.strip():                            # ~40% of rows carry an 'input'
        prompt = f"### Instruction:\n{instr}\n\n### Input:\n{inp}\n\n### Response:\n"
    else:
        prompt = f"### Instruction:\n{instr}\n\n### Response:\n"
    return prompt, out

all_pairs = [format_pair(r) for r in ds["train"].select(range(3000))]
random.shuffle(all_pairs)
train_pairs, val_pairs = all_pairs[:2700], all_pairs[2700:]   # our own held-out split
print(f"train pairs: {len(train_pairs)} | val pairs: {len(val_pairs)}")

README.md:   0%|          | 0.00/7.47k [00:00<?, ?B/s]

data/train-00000-of-00001-a09b74b3ef9c3b(…): reconstructing file:   0%|          |  0.00B / 24.2MB            

data/train-00000-of-00001-a09b74b3ef9c3b(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/52002 [00:00<?, ? examples/s]

train pairs: 2700 | val pairs: 300


### **Hyperparameters:**

In [13]:
EOS = tokenizer.eos_token_id
PAD = tokenizer.pad_token_id
if PAD is None:                              # SmolLM base tokenizer has no pad token
    tokenizer.pad_token = tokenizer.eos_token
    PAD = tokenizer.eos_token_id             # reuse EOS as the pad id
assert EOS is not None and PAD is not None, (EOS, PAD)
print("EOS:", EOS, "| PAD:", PAD)            # confirm both are real ints

MAX_LEN   = 512
batch_sz  = 4
lr, weight_decay, grad_clip = 2e-4, 0.01, 1.0
max_steps, eval_every = 500, 50

EOS: 0 | PAD: 0


### **Masked-example Builder:**

In [14]:
def build_example(prompt_text, response_text):
    p = tokenizer(prompt_text,   add_special_tokens=False)["input_ids"]
    r = tokenizer(response_text, add_special_tokens=False)["input_ids"] + [EOS]
    input_ids = (p + r)[:MAX_LEN]
    labels    = ([-100]*len(p) + r)[:MAX_LEN]      # mask prompt → loss only on response
    return input_ids, labels

### **Collate SFT Batches (Packing) and Batch-Loader:**

In [15]:
def collate(batch_pairs):
    ex = [build_example(p, r) for p, r in batch_pairs]
    maxlen = max(len(ids) for ids, _ in ex)
    input_ids, labels, attn = [], [], []
    for ids, lab in ex:
        pad = maxlen - len(ids)
        input_ids.append(ids + [PAD]  * pad)
        labels.append(   lab + [-100] * pad)       # padding never contributes
        attn.append(     [1]*len(ids) + [0]*pad)   # padding mask
    t = lambda z: torch.tensor(z).to(device)
    return t(input_ids), t(labels), t(attn)

def get_sft_batch(pool):
    return collate(random.sample(pool, batch_sz))

### **Optimizer + Fixed held-out Eval:**

In [16]:
optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

@torch.no_grad()
def estimate_loss(batches=20):
    model.eval()
    total = 0.0
    for _ in range(batches):
        input_ids, labels, attn = get_sft_batch(val_pairs)
        total += model(input_ids=input_ids, attention_mask=attn, labels=labels).loss.item()
    model.train()
    return total / batches

### **Training Loop** (SFT: masked labels + attention_mask):

In [17]:
model.train()
for step in range(max_steps):
    input_ids, labels, attn = get_sft_batch(train_pairs)
    loss = model(input_ids=input_ids, attention_mask=attn, labels=labels).loss

    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
    optimizer.step()

    if step % eval_every == 0 or step == max_steps - 1:
        print(f"step {step:4d} | train {loss.item():.4f} | val {estimate_loss():.4f}")

step    0 | train 2.1963 | val 1.9448
step   50 | train 1.8348 | val 1.7491
step  100 | train 1.7843 | val 1.7212
step  150 | train 1.5239 | val 1.7466
step  200 | train 1.1379 | val 1.5957
step  250 | train 1.4545 | val 1.7582
step  300 | train 1.6243 | val 1.8482
step  350 | train 1.3878 | val 1.5653
step  400 | train 1.5598 | val 1.7261
step  450 | train 1.5839 | val 1.7474
step  499 | train 1.5968 | val 1.6180


### **Generate:**

In [18]:
def sft_generate(instruction, inp=""):
    prompt = (f"### Instruction:\n{instruction}\n\n### Input:\n{inp}\n\n### Response:\n"
              if inp.strip() else
              f"### Instruction:\n{instruction}\n\n### Response:\n")
    model.eval()
    enc = tokenizer(prompt, return_tensors="pt").to(device)
    out = model.generate(**enc, max_new_tokens=150, do_sample=True,
                         temperature=0.7, top_p=0.9, repetition_penalty=1.3,
                         pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True)

print(sft_generate("Explain photosynthesis in simple terms."))

Photosynthesis is the process by which plants convert light energy into chemical energy, providing them with food and oxygen for respiration as well as storing it away until needed again during a long period of darkness or low levels sunlight when most other sources are unavailable (like fire).


### **Base Model:**
Photosynthesis occurs when light energy from the sun is absorbed by chlorophyll molecules within plant cells, resulting in a chemical reaction that turns water and carbon dioxide into glucose (a type of sugar) through an electron transfer process called photophosphorylation or chemiosmosis. This unique method allows plants to convert sunlight directly using their own internal capacity for producing ATP—the powerhouse responsible for generating electricity! By doing so efficiently while maintaining stability inside cell walls like those found on leaves; chloroplasts play essential roles during growth cycles involving multiple stages such as germination/ovulation phases followed closely after fertilization until reaching maturity level where reproduction takes place via sexual means including pollination between male and female reproductive organs involved here before releasing fertilized eggs carrying genetic information necessary later forming new individuals


### **After CPT:**
1 . The answer is a true statement

2  In the process of converting sunlight into food through light-dependent reactions , plants use energy from molecules and chemical bonds to convert carbon dioxide ( CO₂ ) present within air or water vapor onto glucose molecule which then can be used as fuel for cellular respiration by living cells

3 . In order to get more oxygen required during combustion, organisms have evolved specialised structures such as leaves that capture solar radiation at night whilst trapping it inside their bodies while allowing animals consuming them access daily without any risk if they were not able to obtain sufficient amounts needed throughout day time when less efficient mechanisms are available like photolysis [ 9 ] / photosynthesis ; these organs enable greater efficiency towards obtaining O₃ gas necessary via chem



### **After SFT:**
Photosynthesis is the process by which plants and other photosynthetic organisms convert light energy into chemical potential from water to produce glucose (sugar) as a source of fuel for growth, development ,and maintenance .

### **Save The Adapters:**

In [19]:
model.save_pretrained("smollm-lora-sft-alpaca")


---

---

## **RLHF + DPO (Direct Preference Optimization)**

---

---





### **Rebuild the SFT'd model and MERGE the SFT adapters into the base:**

`merge_and_unload` computes `W' = W + (alpha/r)·B·A` for every adapted layer and returns a plain model with no LoRA machinery left.

In [20]:
base  = AutoModelForCausalLM.from_pretrained("HuggingFaceTB/SmolLM-135M").to(device)
sft   = PeftModel.from_pretrained(base, "smollm-lora-sft-alpaca")   # reattach SFT adapters
model = sft.merge_and_unload()                                      # fold W + (α/r)·B·A → W
# `model` is now an ORDINARY CausalLM whose weights ARE the SFT'd SmolLM

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

### **Load + Format the preference dataset** (standard prompt/chosen/rejected):

In [21]:
TEMPLATE = "### Instruction:\n{instruction}\n\n### Response:\n"   # SAME template as SFT

raw = load_dataset("Intel/orca_dpo_pairs", split="train")
print(raw[0].keys())   # dict_keys(['system', 'question', 'chosen', 'rejected'])

def to_pref(row):
    return {
        "prompt":   TEMPLATE.format(instruction=row["question"]),
        "chosen":   row["chosen"],
        "rejected": row["rejected"],
    }

pref = raw.map(to_pref, remove_columns=raw.column_names).select(range(2000))
print(pref[0]["prompt"][:120])
print("CHOSEN  :", pref[0]["chosen"][:80])
print("REJECTED:", pref[0]["rejected"][:80])

README.md:   0%|          | 0.00/196 [00:00<?, ?B/s]

orca_rlhf.jsonl: reconstructing file:   0%|          |  0.00B / 36.3MB            

orca_rlhf.jsonl: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/12859 [00:00<?, ? examples/s]

dict_keys(['system', 'question', 'chosen', 'rejected'])


Map:   0%|          | 0/12859 [00:00<?, ? examples/s]

### Instruction:
You will be given a definition of a task first, then some input of the task.
This task is about using t
CHOSEN  : [
  ["AFC Ajax (amateurs)", "has ground", "Sportpark De Toekomst"],
  ["Ajax You
REJECTED:  Sure, I'd be happy to help! Here are the RDF triplets for the input sentence:




### **LoRA config for the DPO stage + The DPO Hyperparameters:**

In [22]:
peft_config = LoraConfig(
    task_type="CAUSAL_LM", r=8, lora_alpha=16,
    target_modules=["q_proj", "v_proj"], lora_dropout=0.05, bias="none",
)

dpo_config = DPOConfig(
    output_dir                  = "smollm-dpo-orca",
    beta                        = 0.1,        # β — the KL-leash strength from the DPO loss
    learning_rate               = 5e-6,       # tiny, for the reasons from last session
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 4,          # effective batch = 2 × 4 = 8
    num_train_epochs            = 1,
    max_steps                   = 300,        # cap for a fast Colab run
    max_length                  = 1024,
    warmup_steps                = 0.1,
    lr_scheduler_type           = "cosine",
    logging_steps               = 20,
    bf16                        = torch.cuda.is_available(),
    report_to                   = "none",
)

### **Build the DPOTrainer and Train:**

In [23]:
trainer = DPOTrainer(
    model           = model,          # the merged SFT model (plain CausalLM)
    ref_model       = None,           # None → reference = this model with the DPO adapter DISABLED
    args            = dpo_config,
    train_dataset   = pref,
    processing_class= tokenizer,      # TRL's current name for the tokenizer argument
    peft_config     = peft_config,    # fresh adapters, added on top of the merged SFT base
)
trainer.train()

Adding EOS to train dataset:   0%|          | 0/2000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/2000 [00:00<?, ? examples/s]

Dropping fully truncated examples from train dataset:   0%|          | 0/2000 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 0}.


Step,Training Loss
20,0.697590
40,0.695413
60,0.693908
80,0.691928
100,0.679281
120,0.682555
140,0.685178
160,0.675552
180,0.667932
200,0.665353


TrainOutput(global_step=300, training_loss=0.6762592347462972, metrics={'train_runtime': 1978.7944, 'train_samples_per_second': 1.213, 'train_steps_per_second': 0.152, 'total_flos': 1785088616085504.0, 'train_loss': 0.6762592347462972, 'epoch': 1.2152284263959392})

### **Show Log History:**

In [24]:
import pandas as pd
logs = pd.DataFrame(trainer.state.log_history)
cols = [c for c in ["step","loss","rewards/accuracies","rewards/margins",
                    "rewards/chosen","rewards/rejected"] if c in logs]
print(logs[cols].dropna().to_string(index=False))

 step     loss  rewards/accuracies  rewards/margins  rewards/chosen  rewards/rejected
   20 0.697590            0.412500        -0.005796       -0.002038          0.003758
   40 0.695413            0.481250        -0.001896       -0.007607         -0.005711
   60 0.693908            0.500000         0.001356       -0.007432         -0.008788
   80 0.691928            0.575000         0.005886       -0.011791         -0.017676
  100 0.679281            0.643750         0.031284       -0.002463         -0.033747
  120 0.682555            0.593750         0.024926       -0.012482         -0.037408
  140 0.685178            0.612500         0.018897       -0.022245         -0.041142
  160 0.675552            0.625000         0.039183       -0.016035         -0.055218
  180 0.667932            0.712500         0.054121       -0.012007         -0.066128
  200 0.665353            0.775000         0.058970       -0.022622         -0.081592
  220 0.658995            0.775000         0.072528   

### **Save the DPO adapter:**

In [25]:
trainer.save_model("smollm-dpo-orca")

### **Generate & Compare:**

In [26]:
def compare(instruction, max_new_tokens=250, seed=0):
    prompt = TEMPLATE.format(instruction=instruction)
    enc = tokenizer(prompt, return_tensors="pt").to(device)
    m = trainer.model; m.eval()
    kw = dict(max_new_tokens=max_new_tokens, do_sample=True, temperature=0.7,
              top_p=0.9, repetition_penalty=1.3, pad_token_id=tokenizer.eos_token_id)
    strip = lambda o: tokenizer.decode(o[0][enc["input_ids"].shape[1]:], skip_special_tokens=True)

    torch.manual_seed(seed)                          # same random draw for a fair A/B
    with torch.no_grad(), m.disable_adapter():       # DPO adapter OFF → SFT model
        before = strip(m.generate(**enc, **kw))

    torch.manual_seed(seed)                          # reseed → identical sampling stream
    with torch.no_grad():                            # DPO adapter ON → aligned model
        after = strip(m.generate(**enc, **kw))

    print("PROMPT:", instruction)
    print("\n── SFT (adapter OFF, before DPO) ──\n", before)
    print("\n── DPO (adapter ON, after) ──\n", after)

compare("Explain how plants make their food")

PROMPT: Explain how plants make their food

── SFT (adapter OFF, before DPO) ──
 Plants are the primary source of nutrition for most animals, including humans. They take in nutrients from sunlight and water through photosynthesis to produce glucose (food) that is used by cells as energy sources during growth or development stages within a plant's tissues like leaves into new stems,. This process occurs when light shines on green leafy matter at different wavelengths allowing it accesses all necessary components needed throughout life cycle such as carbon dioxide uptake; sugars synthesis using ATP generated via respiration processes leading up towards producing amino acids building blocks required later upon being stored back inside cellular structures called plastids responsible primarily storing lipids fats etc., these molecules undergo further conversion yielding carbohydrates which can be utilized either directly fed onto animal diets where they provide essential protein/amino acid 

### **PROMPT**: Explain why the sky is blue.

### ── **SFT (adapter OFF, before DPO)** ──
 The reason for the color of skies and sunsets being a beautiful shade that we see during sunset or sunrise, as opposed to other colors such as red because it takes longer time than in winter due to its higher temperature level which makes us perceive more blues from this perspective

### ── **DPO (adapter ON, after)** ──
 The reason for the color of skies being seen as they appear to be clear and bluish due to low atmospheric concentrations, which results in a decrease in red light intensity over time because it takes longer at higher altitudes than lower latitudes such as Antarctica or high mountains where sunlight can reach more directly through clouds that are visible from above them (a phenomenon known as the albedo effect).




---



---

## **Push to HuggingFace-Hub**

---



---



In [ ]:
# ═══════════════════════════════════════════════════════════════════
# Assemble the 3 stage models and push each to the Hub
# Run AFTER all adapters are saved: smollm-lora-cpt-wikitext,
# smollm-lora-sft-alpaca, smollm-dpo-orca all exist on disk.
# ═══════════════════════════════════════════════════════════════════
from huggingface_hub import login
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import torch

login()  # paste a WRITE token: hf.co/settings/tokens

BASE = "HuggingFaceTB/SmolLM-135M"
USER = "m-lagnajit"   # ← your HF username

def build_and_push(adapter_chain, repo):
    """Merge base→...→last adapter in order, push standalone model."""
    m = AutoModelForCausalLM.from_pretrained(BASE, torch_dtype=torch.float32)
    for adapter_path in adapter_chain:                 # apply in chain order
        m = PeftModel.from_pretrained(m, adapter_path).merge_and_unload()
    m.push_to_hub(repo)
    print(f"pushed → {repo}")

# Stage 2: base + CPT
build_and_push(["smollm-lora-cpt-wikitext"],
               f"{USER}/minigpt-v3-cpt")

# Stage 3: base + CPT + SFT  (SFT was trained against merged-CPT)
build_and_push(["smollm-lora-cpt-wikitext", "smollm-lora-sft-alpaca"],
               f"{USER}/minigpt-v3-sft")

# Stage 4: base + CPT + SFT + DPO  (DPO trained against merged-SFT)
build_and_push(["smollm-lora-cpt-wikitext", "smollm-lora-sft-alpaca",
                "smollm-dpo-orca"],
               f"{USER}/minigpt-v3-dpo")

# tokenizer (same for all stages) → push once to the DPO repo, reuse everywhere
tok = AutoTokenizer.from_pretrained(BASE)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
tok.push_to_hub(f"{USER}/minigpt-v3-dpo")
print("done — 3 stage models + tokenizer on the Hub")

In [ ]:
# !zip -r "/content/smollm-lora-cpt-wikitext.zip" "/content/smollm-lora-cpt-wikitext"
# !zip -r "/content/smollm-lora-sft-alpaca.zip" "/content/smollm-lora-sft-alpaca"
# !zip -r "/content/smollm-dpo-orca.zip" "/content/smollm-dpo-orca"



---



---

## **RLVR + GRPO**

---



---



### **Load the aligned V3 as the starting policy from HF-Hub:**

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
V3_REPO   = "m-lagnajit/minigpt-v3-dpo"           # deployed V3: base+CPT+SFT+DPO, already merged
policy    = AutoModelForCausalLM.from_pretrained(V3_REPO).to(device)

config.json:   0%|          | 0.00/722 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  538MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

### **Tokenizer:**

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(V3_REPO)
if tokenizer.pad_token is None:                   # SmolLM base has no pad token
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"                   # GRPO generates, so pad left

tokenizer_config.json:   0%|          | 0.00/802 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.52M [00:00<?, ?B/s]

### **FewShots + The prompt-only dataset (note: no responses):**

In [ ]:
import random
from datasets import Dataset
random.seed(0)

FEWSHOT = (
    "### Instruction:\nWhat is 1 + 1?\n\n### Response:\n<answer>2</answer>\n\n"
    "### Instruction:\nWhat is 3 + 4?\n\n### Response:\n<answer>7</answer>\n\n"
    "### Instruction:\nWhat is 5 + 2?\n\n### Response:\n<answer>7</answer>\n\n"
)
TEMPLATE = FEWSHOT + "### Instruction:\n{question}\n\n### Response:\n"

rows = []
for _ in range(400):
    a, b = random.randint(0, 9), random.randint(0, 9)
    rows.append({"question": f"What is {a} + {b}?", "answer": str(a + b)})

prompt_ds = Dataset.from_list([
    {"prompt": TEMPLATE.format(question=r["question"]), "answer": r["answer"]}
    for r in rows
])
assert TEMPLATE.count("### Response:\n<answer>") >= 3, "few-shot exemplars missing"
print(prompt_ds[0]["prompt"])

### Instruction:
What is 1 + 1?

### Response:
<answer>2</answer>

### Instruction:
What is 3 + 4?

### Response:
<answer>7</answer>

### Instruction:
What is 5 + 2?

### Response:
<answer>7</answer>

### Instruction:
What is 6 + 6?

### Response:



### **The Reward Functions:**

In [ ]:
WELL_FORMED = re.compile(r"<answer>\s*(.+?)\s*</answer>", re.DOTALL)  # captures INNER content

def format_reward(completions, **kwargs):
    # Reachable floor — but now requires NON-EMPTY content, so empty <answer></answer> earns 0.
    out = []
    for c in completions:
        m = WELL_FORMED.search(c)
        out.append(1.0 if (m and m.group(1).strip()) else 0.0)
    return out

def accuracy_reward(completions, answer, **kwargs):   # 'answer' = the ground-truth column
    # The stretch signal — only pays if the wrapped answer is actually correct.
    out = []
    for c, gold in zip(completions, answer):
        m = WELL_FORMED.search(c)
        pred = m.group(1).strip() if m else ""
        out.append(1.0 if pred == gold.strip() else 0.0)
    return out

reward_funcs = [format_reward, accuracy_reward]   # summed → max 2.0

### **Config, Trainer, Train & Save the Model:**

In [ ]:
peft_config = LoraConfig(
    task_type="CAUSAL_LM", r=32, lora_alpha=64,
    target_modules=["q_proj","k_proj","v_proj","o_proj"],
    lora_dropout=0.05, bias="none",
)

grpo_config = GRPOConfig(
    output_dir="smollm-grpo-arith",
    learning_rate=5e-5,
    per_device_train_batch_size=8, gradient_accumulation_steps=4,
    num_generations=8, max_completion_length=64,
    temperature=0.9, beta=0.0,
    max_steps=500,
    warmup_steps=10, logging_steps=10,
    bf16=torch.cuda.is_available(), report_to="none",
)

trainer = GRPOTrainer(
    model            = policy,                        # merged aligned V3 (plain CausalLM)
    reward_funcs     = reward_funcs,
    args             = grpo_config,
    train_dataset    = prompt_ds,
    peft_config      = peft_config,                   # fresh GRPO adapters on top
    processing_class = tokenizer,
)
trainer.train()
trainer.save_model("smollm-grpo-format")

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 0}.


Step,Training Loss
10,0.026248
20,-0.022745
30,0.018817
40,0.050838
50,0.062645
60,0.016949
70,0.022993
80,-0.011449
90,0.064053
100,-0.003223


### **Read the training log the right way:**

In [ ]:
logs = pd.DataFrame(trainer.state.log_history)
cols = [c for c in ["step", "reward", "reward_std", "frac_reward_zero_std", "kl", "loss"] if c in logs]
print(logs[cols].dropna().to_string(index=False))   # also: one rewards/<func>/mean column each

 step   reward  reward_std  frac_reward_zero_std      loss
   10 1.084375    0.340581                 0.350  0.026248
   20 1.034375    0.251851                 0.500 -0.022745
   30 1.059375    0.286842                 0.450  0.018817
   40 1.128125    0.360076                 0.425  0.050838
   50 1.137500    0.356620                 0.425  0.062645
   60 1.131250    0.320654                 0.425  0.016949
   70 1.143750    0.377410                 0.350  0.022993
   80 1.175000    0.355399                 0.525 -0.011449
   90 1.140625    0.299197                 0.500  0.064053
  100 1.184375    0.402310                 0.350 -0.003223
  110 1.243750    0.434217                 0.275  0.009061
  120 1.193750    0.364213                 0.500 -0.030970
  130 1.431250    0.473366                 0.325 -0.013137
  140 1.471875    0.462485                 0.375 -0.013568
  150 1.434375    0.424683                 0.650  0.011327
  160 1.506250    0.449603                 0.550 -0.0801

### **Test and Compare:**

In [ ]:
# ── Test the GRPO stage on the VERIFIABLE task: adapter OFF vs ON ──
import re, torch

# Guard that can't false-pass: demands the 3 worked exemplars from Cell 3.
assert TEMPLATE.count("### Response:\n<answer>") >= 3, "Re-run the few-shot arithmetic Cell 3 first."

# SAME content-requiring regex as the reward — empty tags do NOT count.
WELL_FORMED = re.compile(r"<answer>\s*(.+?)\s*</\s*answer\s*>", re.DOTALL | re.IGNORECASE)

# Held-out sums — NONE of these are the exemplars, so this tests generalization.
test = [("What is 3 + 9?", "12"), ("What is 2 + 2?", "4"), ("What is 5 + 1?", "6"),
        ("What is 3 + 6?", "9"), ("What is 0 + 7?", "7")]

policy = trainer.model            # PeftModel: merged V3 + fresh GRPO adapter
policy.eval()
gen_kw = dict(max_new_tokens=64, do_sample=True, temperature=0.7,
              top_p=0.9, repetition_penalty=1.3,
              pad_token_id=tokenizer.eos_token_id)

def run(prompt, seed=0):
    enc   = tokenizer(prompt, return_tensors="pt").to(device)
    strip = lambda o: tokenizer.decode(o[0][enc["input_ids"].shape[1]:], skip_special_tokens=True)
    torch.manual_seed(seed)
    with torch.no_grad(), policy.disable_adapter():   # OFF → few-shot only (pre-GRPO)
        before = strip(policy.generate(**enc, **gen_kw))
    torch.manual_seed(seed)                            # identical draw → fair A/B
    with torch.no_grad():                             # ON → few-shot + GRPO
        after = strip(policy.generate(**enc, **gen_kw))
    return before, after

def grade(text, gold):
    match = WELL_FORMED.search(text)
    pred  = re.sub(r"[^0-9-]", "", match.group(1)) if match else ""   # pull the integer out
    return bool(pred), (pred == gold), pred

# Sanity: the exemplars MUST be visible before the real question
print("PROMPT FED TO MODEL:\n" + "-" * 60)
print(TEMPLATE.format(question=test[0][0]))
print("=" * 60)

fb = ab = fa = aa = 0     # format/acc, before/after
for q, gold in test:
    before, after = run(TEMPLATE.format(question=q))
    f_b, a_b, _ = grade(before, gold)
    f_a, a_a, _ = grade(after,  gold)
    fb += f_b; ab += a_b; fa += f_a; aa += a_a
    print(f"Q: {q}  (gold {gold})")
    print(f"  before [fmt={f_b} acc={a_b}] -> {before.strip()[:80]}")
    print(f"  after  [fmt={f_a} acc={a_a}] -> {after.strip()[:80]}")
    print("-" * 60)

print(f"\nformat   — before: {fb}/5   after: {fa}/5")
print(f"accuracy — before: {ab}/5   after: {aa}/5")

PROMPT FED TO MODEL:
------------------------------------------------------------
### Instruction:
What is 1 + 1?

### Response:
<answer>2</answer>

### Instruction:
What is 3 + 4?

### Response:
<answer>7</answer>

### Instruction:
What is 5 + 2?

### Response:
<answer>7</answer>

### Instruction:
What is 3 + 9?

### Response:

Q: What is 3 + 9?  (gold 12)
  before [fmt=True acc=False] -> <answer><value_str="8">60</value\_str></answer>.
  after  [fmt=True acc=False] -> <answer>10.8</answer>
------------------------------------------------------------
Q: What is 2 + 2?  (gold 4)
  before [fmt=False acc=False] -> <answer><value_str="8">6</value\_str></ans>')
  after  [fmt=True acc=True] -> <answer>4</ Answer >
------------------------------------------------------------
Q: What is 5 + 1?  (gold 6)
  before [fmt=True acc=False] -> <answer><value_str="8">60</value\_str></answer>.
  after  [fmt=True acc=True] -> <answer>6</ Answer >
---------------------------------------------------------

In [ ]:
from collections import Counter
def bucket(after, gold):
    m = WELL_FORMED.search(after)
    pred = re.sub(r"[^0-9-]", "", m.group(1)) if m else ""
    fmt = bool(m and pred)
    acc = (pred == gold)
    return (1.0 if fmt else 0.0) + (1.0 if acc else 0.0)   # → 0.0, 1.0, or 2.0

dist = Counter(bucket(run(TEMPLATE.format(question=q))[1], gold) for q, gold in test)
print("reward buckets (after):", dict(dist))   # e.g. {1.0: 4, 0.0: 1} = all format, no accuracy

reward buckets (after): {1.0: 1, 2.0: 4}


### **Zip & Save:**

In [ ]:
!zip -r "/content/smollm-grpo-arith.zip" "/content/smollm-grpo-arith"
!zip -r "/content/smollm-grpo-format.zip" "/content/smollm-grpo-format"

  adding: content/smollm-grpo-arith/ (stored 0%)
  adding: content/smollm-grpo-arith/checkpoint-500/ (stored 0%)
  adding: content/smollm-grpo-arith/checkpoint-500/adapter_config.json (deflated 60%)
  adding: content/smollm-grpo-arith/checkpoint-500/optimizer.pt (deflated 8%)
  adding: content/smollm-grpo-arith/checkpoint-500/rng_state.pth (deflated 26%)
  adding: content/smollm-grpo-arith/checkpoint-500/trainer_state.json (deflated 87%)
  adding: content/smollm-grpo-arith/checkpoint-500/adapter_model.safetensors (deflated 7%)
  adding: content/smollm-grpo-arith/checkpoint-500/tokenizer_config.json (deflated 61%)
  adding: content/smollm-grpo-arith/checkpoint-500/scheduler.pt (deflated 61%)
  adding: content/smollm-grpo-arith/checkpoint-500/training_args.bin (deflated 55%)
  adding: content/smollm-grpo-arith/checkpoint-500/README.md (deflated 65%)
  adding: content/smollm-grpo-arith/checkpoint-500/tokenizer.json (deflated 82%)
  adding: content/smollm-grpo-arith/README.md (deflated 47%

### **Push to HuggingFace-Hub:**

In [ ]:
from huggingface_hub import login
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import torch

login()
USER = "m-lagnajit"

# Start from the ALREADY-MERGED v3-dpo on the Hub (= base+CPT+SFT+DPO), not local folders.
m = AutoModelForCausalLM.from_pretrained(f"{USER}/minigpt-v3-dpo", torch_dtype=torch.float32)
# Attach the one adapter from THIS session and fold it in.
m = PeftModel.from_pretrained(m, "smollm-grpo-arith").merge_and_unload()
m.push_to_hub(f"{USER}/minigpt-v3-grpo")

tok = AutoTokenizer.from_pretrained(f"{USER}/minigpt-v3-dpo")
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
tok.push_to_hub(f"{USER}/minigpt-v3-grpo")
print("done → minigpt-v3-grpo")

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...4tsfpon/model.safetensors:   0%|          | 12.0kB /  538MB            

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

done → minigpt-v3-grpo


### **Sanity Check:**

In [ ]:
check = AutoModelForCausalLM.from_pretrained(f"{USER}/minigpt-v3-grpo").to(device).eval()
enc = tokenizer(TEMPLATE.format(question="What is 6 + 2?"), return_tensors="pt").to(device)
out = check.generate(**enc, max_new_tokens=64, do_sample=True, temperature=0.7,
                     top_p=0.9, repetition_penalty=1.3, pad_token_id=tokenizer.eos_token_id)
print(tokenizer.decode(out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True))

config.json:   0%|          | 0.00/722 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  538MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

<answer>8</ answer><br /> 

